# Step 07 â€” LLM Error Row Re-run

**Input:**
- `data/output/llm_predictions/prediction_errors.parquet` â€” rows that failed validation in nb06
- `data/output/05_condensed_buildings_with_pois.gpkg` â€” for rebuilding sentences

**Output:** Appended to `data/output/llm_predictions/predictions_checkpoint.parquet`

**From linux branch:** This notebook handles the multi-day rerun workflow with checkpoint/resume and parallel workers.
Run it repeatedly until `prediction_errors.parquet` is empty (or a stable minimum).

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import json
import os
import re
import time
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

load_dotenv(dotenv_path=ROOT / '.env')
TU_TOKEN = os.getenv('TU_KI_TOOLBOX_TOKEN')
if not TU_TOKEN:
    raise RuntimeError('Missing TU_KI_TOOLBOX_TOKEN')

print('Config loaded')

## Load error rows and rebuild sentences

In [ ]:
if not LLM_ERRORS_FILE.exists():
    print('No error file found â€” nothing to rerun.')
else:
    error_df = pd.read_parquet(LLM_ERRORS_FILE)
    print(f'Error rows to rerun: {len(error_df):,}')

    # Rebuild sentences from the condensed buildings file
    pois = gpd.read_file(CONDENSED_BUILDINGS_FILE)
    pois_map = pois.set_index('gml_id').to_dict('index')

    def row_to_llm_input(row_dict):
        def is_missing(x):
            if x is None: return True
            if isinstance(x, float) and pd.isna(x): return True
            if isinstance(x, str) and x.strip().lower() in ('', 'none', 'nan', 'null'): return True
            return False
        def fmt(x):
            if is_missing(x): return None
            if isinstance(x, (list, tuple, set)):
                v = [str(i).strip() for i in x if not is_missing(i)]
                return '; '.join(v) if v else None
            return str(x).strip()
        precise = [('osm_names','name'),('amenity','amenity'),('building','building'),
                   ('shop','shop'),('tourism','tourism'),('information','information'),
                   ('website','website'),('email','email')]
        general = [('label_en','building_label'),('osm_building_type','osm_building_type'),
                   ('osm_landuse_class','osm_landuse_class'),('osm_landuse_name','osm_landuse_name'),
                   ('gfk_class','gfk_class'),('ALKIS_Landuse_info','alkis_landuse'),
                   ('tags_search','tags'),('additional_information','additional_info')]
        def collect(fields):
            return [f'{lbl}={fmt(row_dict.get(col))}' for col, lbl in fields if fmt(row_dict.get(col))]
        parts = []
        p = collect(precise); g = collect(general)
        if p: parts.append('precise_known_info: ' + ' | '.join(p))
        if g: parts.append('general_building_context: ' + ' | '.join(g))
        return '\n'.join(parts)

    error_df['sentence'] = error_df['gml_id'].apply(
        lambda gid: row_to_llm_input(pois_map.get(str(gid), {}))
    )

## System prompt, LLM call, and validation (same as notebook 06)

In [ ]:
# Helpers and validation imported from llm_utils (single source of truth)
from llm_utils import extract_first_json
from llm_utils import validate as _validate

def validate(obj):
    _validate(obj, TARGET_MID_LABELS)

SYSTEM_PROMPT = """
You are a building activity interpreter and classifier.
Classify the building into activity labels (mid_labels) and a Bosserhof building-use class.
ALLOWED ACTIVITY LABELS: work, university, school, childcare, retail_daily, retail_non_daily, leisure, sports, errands, meetup, lessons, business
BOSSERHOF CATEGORIES: Transport | Yards/depots/storage | Industrial operations/Production | Crafts and trades | Services | Retail | Public facilities | Facilities for culture/leisure/sports
Return null for bosserhof_class if the place is not an enterable building.
OUTPUT: strict JSON only: {"interpreted_type": "...", "mid_labels": [...], "bosserhof_class": "...", "reason": "..."}
""".strip()

def call_tu_llm(text):
    headers = {"Authorization": f"Bearer {TU_TOKEN}", "Accept": "application/json", "Content-Type": "application/json"}
    payload = {"thread": None, "prompt": text, "model": LLM_MODEL,
               "customInstructions": SYSTEM_PROMPT, "hideCustomInstructions": True,
               "reasoning": {"effort": LLM_REASONING}}
    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            r = requests.post(LLM_API_URL, headers=headers, json=payload, stream=True, timeout=LLM_TIMEOUT_SEC)
            r.raise_for_status()
            full = ""
            for line in r.iter_lines(decode_unicode=True):
                if not line: continue
                try: ev = json.loads(line)
                except: continue
                if ev.get("type") == "chunk": full += ev.get("content", "")
                elif ev.get("type") == "done": full = ev.get("response", full); break
            return full
        except Exception as e:
            last = e; time.sleep(LLM_BACKOFF_SEC * attempt)
    raise RuntimeError(f"LLM failed: {last}")

def predict_row(gml_id, sentence):
    try:
        raw = call_tu_llm(str(sentence) if sentence else "")
        obj = extract_first_json(raw); validate(obj)
        return {"gml_id": gml_id, "interpreted_type": obj["interpreted_type"],
                "mid_labels": obj["mid_labels"], "bosserhof_class": obj["bosserhof_class"],
                "short_reason": obj["reason"], "error": None}
    except Exception as e:
        return {"gml_id": gml_id, "interpreted_type": "error",
                "mid_labels": [], "bosserhof_class": None, "short_reason": None, "error": str(e)}


## Rerun error rows with checkpoint/resume

In [ ]:
if not LLM_ERRORS_FILE.exists():
    print('No errors to rerun.')
else:
    # Skip rows already recovered in a previous partial rerun
    done_ids = set()
    if LLM_CHECKPOINT_FILE.exists():
        done_ids = set(pd.read_parquet(LLM_CHECKPOINT_FILE)['gml_id'].astype(str))

    todo = error_df[~error_df['gml_id'].astype(str).isin(done_ids)].copy()
    print(f'Error rows to retry: {len(todo):,}')

    def append_parquet(path, rows):
        new_df = pd.DataFrame(rows)
        if path.exists():
            existing = pd.read_parquet(path)
            combined = pd.concat([existing, new_df], ignore_index=True).drop_duplicates(subset='gml_id', keep='last')
        else:
            combined = new_df
        combined.to_parquet(path, index=False)

    n = len(todo)
    chunks = [todo.iloc[i * LLM_CHUNK_SIZE:(i + 1) * LLM_CHUNK_SIZE] for i in range(max(1, n // LLM_CHUNK_SIZE))]
    if n % LLM_CHUNK_SIZE: chunks.append(todo.iloc[(n // LLM_CHUNK_SIZE) * LLM_CHUNK_SIZE:])

    new_errors = []
    for i, chunk in enumerate(chunks):
        results = []
        with ThreadPoolExecutor(max_workers=LLM_MAX_WORKERS) as ex:
            futures = {ex.submit(predict_row, row['gml_id'], row.get('sentence', '')): idx
                       for idx, row in chunk.iterrows()}
            for future in as_completed(futures):
                results.append(future.result())

        good   = [r for r in results if r['error'] is None]
        errors = [r for r in results if r['error'] is not None]

        if good:   append_parquet(LLM_CHECKPOINT_FILE, good)
        new_errors.extend(errors)
        print(f'Chunk {i+1}/{len(chunks)} | recovered={len(good)} | still_failing={len(errors)}')

    # Overwrite error file with only the remaining failures
    if new_errors:
        pd.DataFrame(new_errors).to_parquet(LLM_ERRORS_FILE, index=False)
        print(f'{len(new_errors)} rows still failing â€” run this notebook again or investigate')
    else:
        LLM_ERRORS_FILE.unlink(missing_ok=True)
        print('All error rows resolved!')